# Testing the Capstone Agent

The 20-case benchmark of the capstone chapter is the floor, not the ceiling. This notebook puts the finished agent on a test stand and asks how it behaves under realistic input variation, and where it breaks. The method is the companion volume's `gmstest` framework applied end to end to one governed agent; the numbers here are read from the pinned campaign so they match the chapter exactly.

**Reference (run separately):** the framework calls that produced these numbers. The agent SUT wraps `build_complaint_harness` unchanged; the probes read the deployed gates and the store's geometry directly.

```python
from gmstest import resolve, compose, evaluate
from apps.complaint_sut import (
    AgentSUT, score_components, retrieval_benchmark,
    probe_policy_gate, probe_plausibility_gate,
    probe_value_polarity, probe_provenance_grounding,
)
sut  = AgentSUT()                                  # build_complaint_harness, unchanged
cat, items = resolve.load_catalog('complaint')
suite = resolve.suite(cat, factors=['clarity', 'entity_aliasing', 'reasoning_cue'])
scns  = compose(suite, items, n_runs=120, seed=42, balance_base=True)
rows  = evaluate.run(scns, sut, workflow_order=WORKFLOW)
```

**What the next cell does** — loads the pinned campaign artifacts:

1. **Find the data root.** Walk up from `Path.cwd()` until `data/capstone_run.json` is found.
2. **Load them.** `D` (campaign + gates + attribution + per-tool), `R` (graph-truth retrieval), `C` (draft groundedness, value-polarity, provenance, resilience), and the per-run `rows`.

In [ ]:
import json
from pathlib import Path

root = Path.cwd()
while not (root/'data'/'capstone_run.json').exists() and root != root.parent:
    root = root.parent
D = json.loads((root/'data'/'capstone_run.json').read_text())
R = json.loads((root/'data'/'capstone_retrieval.json').read_text())
C = json.loads((root/'data'/'capstone_companions.json').read_text())
rows = json.loads((root/'data'/'capstone_rows.json').read_text())['rows']
WORKFLOW = ['classify_complaint', 'extract_facts', 'search_policy',
            'flag_regulatory', 'draft_response']
print('loaded artifacts from', root/'data')

## Designing the complaint suite

The unit of coverage is a complaint *scenario*: one of the twenty seed cases under a controlled combination of three presentation factors (`clarity`, `entity_aliasing`, `reasoning_cue`). The seed case is a *blocking* factor, balanced across the design but excluded from the attribution model. The design is a space-filling Sobol sequence refined to balance the blocking factor: the seed is represented exactly six times each, the presentation levels cover the joint space without empty cells.

**What the next cell does** — reports the design coverage: the per-level counts of each presentation factor and the per-seed balance, read from the pinned rows.

In [ ]:
from collections import Counter
print(f'{len(rows)} scenarios')
for f in ('f_clarity', 'f_entity_aliasing', 'f_reasoning_cue'):
    c = Counter(r[f] for r in rows)
    print(f'  {f[2:]:16s} {dict(c.most_common())}')
seeds = Counter(r['seed'] for r in rows)
print(f'  seed_case        {len(seeds)} cases, {min(seeds.values())} each')

The seed balance is exact because it is the blocking factor; the presentation levels are not equal-frequency, because the Sobol design fills the joint factor space rather than each margin, and the logistic attribution handles the unequal cells directly. Each row is then materialized into a message: the `clarity` dimension is rephrased by the local Qwen3-4B model, and the mechanical factors (aliasing, reasoning cue) are stamped on after, so the model cannot correct an injected typo or drop a coaxing nudge. The dollar amount survives every transform: it is the claim the judgment stage checks.

## Running the agent, measuring the whole trajectory

Each message runs through the agent exactly as the capstone built it, returning a full `Trajectory`, not a label. The stand reads it at three levels: the per-query **decision** (right classification *and* right escalate/don't-escalate call), **trajectory structure** (workflow adherence, escalation path) and **process health** (the audit chain). The per-tool components are scored separately, not folded into the decision.

**What the next cell does** — prints the campaign summary: decision accuracy, workflow adherence and whether the audit chain verifies.

In [ ]:
print(f"{D['n_runs']} scenarios")
print(f"  decision accuracy:   {D['overall']['accuracy']:.3f}")
print(f"  workflow adherence:  {D['workflow_adherence']:.3f}")
print(f"  audit verifies:      {D['audit_verifies']}")

The agent reaches the right decision on eighty-one of the hundred and twenty scenarios (**0.675**). Workflow adherence is perfect: the plausibility gate held the five-step sequence in order on every phrasing, so no failure is the agent firing tools out of turn. The audit chain verifies across the campaign, so any failure surfaced elsewhere is behavioral, not an integrity failure of the record. The next stages ask what the agent got wrong on the runs it missed, and why.

## Judgment: groundedness and provenance

Even a correct decision leaves the safety-relevant question open: is the draft grounded in policy or does it just sound like it? The stand reuses groundedness-as-distance. The draft's fee claim is aligned to a store triple `(overdraft, has_fee_amount, 35.0)` and scored with `score_triple`; the tier bands are calibrated per relation from the store, not hard-coded.

**What the next cell does** — prints the judged example drafts (`C['draft_groundedness']['examples']`): the aligned triple, its geodesic distance and the resulting tier.

In [ ]:
dg = C['draft_groundedness']
print(f"{'draft':26s}{'triple':44s}{'distance':>9s}  tier")
for e in dg['examples']:
    print(f"{e['label']:26s}{e['triple']:44s}{e['geodesic']:>9.3f}  {e['tier']}")

The committed overdraft fee scores a geodesic distance near **0.06** and lands grounded; a fabricated \$50 fee scores near **1.55** and lands in the fabrication band. Run across the suite, this pairs with a **provenance** check that traces each claim back to the policy text.

**What the next cell does** — prints the provenance grounding (`C['provenance_grounding']`): how many verifiable claims carry a source sentence, and whether the cited span contains the value.

In [ ]:
pg = C['provenance_grounding']
print(f"claims with a source sentence : {pg['sentence_provenance_n']}/{pg['facts']}"
      f"  ({pg['sentence_provenance_rate']:.2f})")
print(f"cited span contains the value : {pg['consistent_n']}/{pg['facts']}"
      f"  ({pg['consistent_rate']:.2f})")

Every one of the thirty-three verifiable claims cites a span that contains the value it states (consistency **1.00**); twenty-six resolve to a sentence-level span in the prose, the rest to a table cell. No draft silently states a wrong number, and each stated number is traceable to the sentence of policy it came from.

## Attribution: which input drives the failures

Binary rates do not say which factor caused the failures. The framework fits a logistic regression of `correct` on the three presentation factors, runs an analysis of deviance and applies Benjamini--Hochberg correction. The seed case is excluded (a blocking factor, not a property under test).

**What the next cell does** — prints the corrected attribution table (`D['overall']['attribution']`): per factor the deviance `G2`, adjusted p-value and pseudo-$R^2$, with a `*` on any factor significant after correction.

In [ ]:
for r in D['overall']['attribution']:
    flag = '*' if r['significant_adj'] else ' '
    print(f" {flag} {r['factor']:16s} p_adj={r['p_adj']:.4f}  pseudo_r2={r['pseudo_r2']:.3f}")

No factor is significant after correction. The nearest is `clarity` ($p_{adj}=0.96$): the empirical rates tilt slightly toward the `ambiguous` level, but the effect does not separate from noise. This reverses the earlier agent, where a downplaying cue was the dominant, significant driver: on the retrained classifier and the operator-native retriever, none of the presentation factors moves the decision. The remaining failures are few and spread, and the place to look is the per-component decomposition, not the input factors.

## Which component failed: per-tool decomposition

Factor attribution names the *input*; here it named none. The weak-link analysis names the *component*: each tool is scored against its own ground truth, and each wrongly-decided run is charged to the first tool, in workflow order, that erred. The two label-scored tools read from `D['per_tool']`; `search_policy` is graded against the graph ground truth (`R`), and `extract_facts` shows up through the two downstream tools it drives, not a row of its own.

**What the next cell does** — prints the two label-scored tool accuracies, the graph-truth retrieval recall/precision (with parse and bind rates), and the weak-link blame counts.

In [ ]:
for tool in ('classify_complaint', 'flag_regulatory'):
    s = D['per_tool'][tool]
    print(f"  {tool:18s} accuracy={s['accuracy']:.2f}  ({s['correct']}/{s['scored']})")
g = R['gms']
print(f"  search_policy      recall={g['recall']:.2f}  precision={g['precision']:.2f}"
      f"  (parse {g['parse_rate']:.2f}, bind {g['bind_rate']:.2f})")
print(f"  dense baseline     recall={R['dense']['recall']:.2f}  miss stages {R['miss_stage']}")
print(f"  weak-link blame: {D['weak_link']}")

The retrieval row is graded against the **graph ground truth**, not a top-$k$ of policy labels: a hit requires the retrieved *value* to match the value the graph holds, and every miss localizes to parse, bind or retrieve. On that measure the operator-native retriever reaches recall and precision **0.85** with parse and bind rates **1.00** --- every question bound to the correct policy entity --- against a dense baseline of 0.65. All three misses are at the retrieve stage (the right policy, an adjacent field), so what is left is relation disambiguation, not retrieval. The three components are each sound; the weak-link blame is sparse (nine to the classifier, one to the flagger) and the rest of the wrong runs turn on the escalation decision, a whole-trajectory property.

## Resilience: faulting each tool in turn

The complementary question to correctness is what happens when a tool *fails*. The real `ToolGateway` is installed on the executor and, one tool at a time, a `FaultProfile` makes that tool error. A governed agent must fail loud --- escalate --- never silently pass a corrupted result downstream.

**What the next cell does** — prints the per-tool detection rate and silent-failure count from the pinned resilience block (`C['resilience']`).

In [ ]:
res = C['resilience']
print(f"fault injected : {res['fault']}   overall detection {res['detection_rate']:.2f}"
      f"   silent failures {res['silent_failures']}")
for tool, rate in res['per_tool_detection'].items():
    print(f"  {tool:18s} detection_rate={rate:.2f}")

Under a hard fault on any tool the agent fails loud and escalates (detection **1.0** everywhere) and never silently proceeds with a corrupted result (silent **0**). Each gateway decision is recorded, so the test is auditable.

## The hardest fault: a reversed stance

The hardest fault is a plausible-but-wrong *structured* result: a drafted reply that **inverts** a policy's stance --- asserting *optional* where the policy makes it required, or *permitted* where it is forbidden. Type and shape are correct, so a syntactic check passes; only a comparison against the stored stance catches it. The value-polarity verifier returns `supported`, `contradicted` or `uncertain`.

**What the next cell does** — prints the value-polarity result (`C['value_polarity']`): how many stance reversals are flagged contradicted, and how many true-stance synonyms are accepted.

In [ ]:
vp = C['value_polarity']
print(f"stance reversals flagged contradicted : {vp['reversal_detected']}/{vp['reversal_n']}"
      f"  ({vp['reversal_rate']:.2f})")
print(f"synonym stances accepted as supported : {vp['synonym_accepted']}/{vp['synonym_n']}"
      f"  ({vp['synonym_rate']:.2f})")

On a four-item reversal battery the verifier flags three of four reversals as contradicted; the miss is a required-to-optional assertion scored `uncertain`, a deferral to review rather than a false pass. It accepts three of four true-stance synonyms. The stance reversal is the plausible-but-wrong fault in its purest form, caught downstream by the same substrate the retriever and draft judge consult, not at the tool boundary.

**What the next cell does** — asserts every reported number against the pinned artifacts so the notebook, the chapter and the JSON stay in agreement.

In [ ]:
# Campaign
assert D['n_runs'] == 120
assert D['overall']['accuracy'] == 0.675
assert D['workflow_adherence'] == 1.0 and D['audit_verifies'] is True
# Attribution: no factor significant
assert not any(r['significant_adj'] for r in D['overall']['attribution'])
assert next(r for r in D['overall']['attribution'] if r['factor'] == 'clarity')['p_adj'] > 0.9
# Per-tool + graph-truth retrieval
assert D['per_tool']['classify_complaint']['accuracy'] == 0.902
assert D['per_tool']['flag_regulatory']['accuracy'] == 0.944
assert R['gms']['recall'] == 0.85 and R['gms']['precision'] == 0.85
assert R['gms']['parse_rate'] == 1.0 and R['gms']['bind_rate'] == 1.0
assert set(R['miss_stage']) == {'retrieve'}
assert D['weak_link'] == {'flag_regulatory': 1, 'classify_complaint': 9}
# Draft groundedness + provenance
ex = {e['label']: e for e in C['draft_groundedness']['examples']}
assert ex['grounded overdraft fee']['tier'] == 'grounded'
assert abs(ex['grounded overdraft fee']['geodesic'] - 0.056) < 1e-2
assert ex['distorted overdraft fee']['tier'] == 'fabrication'
assert C['provenance_grounding']['consistent_rate'] == 1.0
# Resilience + stance
assert C['resilience']['detection_rate'] == 1.0 and C['resilience']['silent_failures'] == 0
assert C['value_polarity']['reversal_detected'] == 3 and C['value_polarity']['reversal_n'] == 4
print('self-check passed')